# 01 - Introducao ao ETS (Error, Trend, Seasonal) - SOLUCAO

Este notebook apresenta os fundamentos dos modelos de suavizacao exponencial usando a biblioteca **chronobox**, com **todos os exercicios resolvidos**.

**Conteudo:**
1. Taxonomia ETS: os 30 modelos possiveis
2. ETS(A,N,N) - Suavizacao Exponencial Simples
3. ETS(A,A,N) - Metodo de Holt (tendencia linear)
4. ETS(A,A,A) - ETS completo com sazonalidade
5. Decomposicao em componentes (nivel, tendencia, sazonalidade)
6. Interpretacao dos parametros de suavizacao
7. **Exercicios resolvidos**

## 1. Taxonomia ETS: 30 Modelos

O framework **ETS** (Error, Trend, Seasonal) classifica modelos de suavizacao exponencial por tres componentes:

- **Error (E):** Aditivo (A) ou Multiplicativo (M)
- **Trend (T):** Nenhum (N), Aditivo (A), Aditivo Amortecido (Ad), Multiplicativo (M), Multiplicativo Amortecido (Md)
- **Seasonal (S):** Nenhum (N), Aditivo (A), Multiplicativo (M)

Isso resulta em $2 \times 5 \times 3 = 30$ combinacoes possiveis.

### Casos Especiais Conhecidos

| Modelo ETS | Nome Classico |
|------------|---------------|
| ETS(A,N,N) | Suavizacao Exponencial Simples (SES) |
| ETS(A,A,N) | Metodo de Holt (tendencia linear) |
| ETS(A,Ad,N) | Metodo de Holt com tendencia amortecida |
| ETS(A,A,A) | Holt-Winters aditivo |
| ETS(M,A,M) | Holt-Winters multiplicativo |

## 2. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import json

from chronobox import ETS
from chronobox.models.ets import get_all_ets_models
from scipy.stats import norm, probplot

np.random.seed(42)

# Diretorios
BASE_DIR = os.path.join(os.path.dirname(os.getcwd()))
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('chronobox importado com sucesso!')
print(f'Total de modelos ETS possiveis: {len(get_all_ets_models())}')

In [ ]:
# Carregar dataset airline (Box-Jenkins, passageiros mensais 1949-1960)
airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)
y_airline = airline['passengers'].values.astype(float)

# Carregar dataset sintetico ETS
ets_synth = pd.read_csv(os.path.join(DATA_DIR, 'ets_synthetic.csv'), parse_dates=['date'])
ets_synth.set_index('date', inplace=True)
y_synth = ets_synth['value'].values.astype(float)

print(f'Airline: {len(y_airline)} obs, periodo {airline.index[0].strftime("%Y-%m")} a {airline.index[-1].strftime("%Y-%m")}')
print(f'Sintetico: {len(y_synth)} obs, periodo {ets_synth.index[0].strftime("%Y-%m")} a {ets_synth.index[-1].strftime("%Y-%m")}')

# Visualizacao inicial
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y_airline, color='steelblue')
axes[0].set_title('Airline Passengers (1949-1960)')
axes[0].set_ylabel('Passageiros (milhares)')
axes[1].plot(y_synth, color='darkorange')
axes[1].set_title('Serie Sintetica ETS(M,A,M)')
axes[1].set_ylabel('Valor')
plt.tight_layout()
plt.show()

## 3. ETS(A,N,N) - Suavizacao Exponencial Simples

O modelo mais simples: **sem tendencia, sem sazonalidade**, apenas erro aditivo.

Equacoes de estado:
$$y_t = l_{t-1} + \varepsilon_t$$
$$l_t = l_{t-1} + \alpha \varepsilon_t$$

O parametro $\alpha \in (0,1)$ controla a velocidade de adaptacao:
- $\alpha \to 0$: previsao quase constante (media historica)
- $\alpha \to 1$: previsao = ultima observacao (random walk)

In [ ]:
# Ajustar ETS(A,N,N) - Suavizacao Exponencial Simples
model_ann = ETS(error='A', trend='N', seasonal='N')
res_ann = model_ann.fit(y_airline)

print(res_ann.summary())
print(f'\nParametro alpha: {res_ann.alpha:.4f}')
print(f'Nivel inicial (l0): {res_ann.l0:.2f}')
print(f'AIC: {res_ann.aic:.2f}')

In [ ]:
# Grafico: Ajuste ETS(A,N,N)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(res_ann.fittedvalues, label='ETS(A,N,N) ajustado', color='darkorange',
        linewidth=1.2, linestyle='--')
ax.set_title('ETS(A,N,N) - Suavizacao Exponencial Simples')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('Nota: O ETS(A,N,N) nao captura tendencia nem sazonalidade.')
print('A previsao sera uma linha constante (nivel atual).')

## 4. ETS(A,A,N) - Metodo de Holt (Tendencia Linear)

Adiciona um componente de **tendencia aditiva**:

$$y_t = l_{t-1} + b_{t-1} + \varepsilon_t$$
$$l_t = l_{t-1} + b_{t-1} + \alpha \varepsilon_t$$
$$b_t = b_{t-1} + \beta \varepsilon_t$$

In [ ]:
# Ajustar ETS(A,A,N) - Holt com tendencia linear
model_aan = ETS(error='A', trend='A', seasonal='N')
res_aan = model_aan.fit(y_airline)

print(res_aan.summary())
print(f'\nalpha (nivel): {res_aan.alpha:.4f}')
print(f'beta (tendencia): {res_aan.beta:.4f}')
print(f'Nivel inicial (l0): {res_aan.l0:.2f}')
print(f'Tendencia inicial (b0): {res_aan.b0:.2f}')
print(f'AIC: {res_aan.aic:.2f}')

In [ ]:
# Comparacao ETS(A,N,N) vs ETS(A,A,N)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(res_ann.fittedvalues, label=f'ETS(A,N,N) AIC={res_ann.aic:.1f}',
        color='darkorange', linewidth=1.2, linestyle='--')
ax.plot(res_aan.fittedvalues, label=f'ETS(A,A,N) AIC={res_aan.aic:.1f}',
        color='green', linewidth=1.2, linestyle='--')
ax.set_title('Comparacao: sem tendencia vs com tendencia')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('O ETS(A,A,N) captura a tendencia crescente, mas ainda ignora a sazonalidade.')

## 5. ETS(A,A,A) - Modelo Completo com Sazonalidade Aditiva

$$y_t = l_{t-1} + b_{t-1} + s_{t-m} + \varepsilon_t$$
$$l_t = l_{t-1} + b_{t-1} + \alpha \varepsilon_t$$
$$b_t = b_{t-1} + \beta \varepsilon_t$$
$$s_t = s_{t-m} + \gamma \varepsilon_t$$

In [ ]:
# Ajustar ETS(A,A,A) - Modelo completo com sazonalidade aditiva
model_aaa = ETS(error='A', trend='A', seasonal='A', seasonal_period=12)
res_aaa = model_aaa.fit(y_airline)

print(res_aaa.summary())
print(f'\nalpha (nivel): {res_aaa.alpha:.4f}')
print(f'beta (tendencia): {res_aaa.beta:.4f}')
print(f'gamma (sazonalidade): {res_aaa.gamma:.4f}')
print(f'AIC: {res_aaa.aic:.2f}')

In [ ]:
# Decomposicao em componentes do ETS(A,A,A)
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(y_airline, label='Observado', color='steelblue')
axes[0].plot(res_aaa.fittedvalues, label='Ajustado', color='darkorange', linestyle='--')
axes[0].set_title('ETS(A,A,A) - Airline Passengers')
axes[0].legend()
axes[0].set_ylabel('Passageiros')

states = res_aaa.states
n = len(y_airline)
axes[1].plot(states[:n, 0], color='green')
axes[1].set_title('Componente: Nivel ($l_t$)')
axes[1].set_ylabel('Nivel')

axes[2].plot(states[:n, 1], color='purple')
axes[2].set_title('Componente: Tendencia ($b_t$)')
axes[2].set_ylabel('Tendencia')

axes[3].plot(states[:n, 2], color='crimson')
axes[3].set_title('Componente: Sazonalidade ($s_t$)')
axes[3].set_ylabel('Sazonal')
axes[3].set_xlabel('Observacao')

plt.tight_layout()
plt.show()

## 6. Comparacao e Interpretacao dos Parametros

In [ ]:
# Comparacao de todos os modelos ajustados
print(f'{"Modelo":<15} {"alpha":>8} {"beta":>8} {"gamma":>8} {"AIC":>10} {"BIC":>10}')
print('-' * 55)

modelos = {
    'ETS(A,N,N)': res_ann,
    'ETS(A,A,N)': res_aan,
    'ETS(A,A,A)': res_aaa,
}

for nome, res in modelos.items():
    alpha = f'{res.alpha:.4f}'
    beta = f'{res.beta:.4f}' if res.beta is not None else '   -   '
    gamma = f'{res.gamma:.4f}' if res.gamma is not None else '   -   '
    print(f'{nome:<15} {alpha:>8} {beta:>8} {gamma:>8} {res.aic:>10.2f} {res.bic:>10.2f}')

melhor = min(modelos, key=lambda k: modelos[k].aic)
print(f'\nMelhor modelo por AIC: {melhor}')

## 7. Diagnostico de Residuos

In [ ]:
# Diagnostico de residuos do ETS(A,A,A)
resid = res_aaa.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(resid, color='steelblue', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Residuos ao Longo do Tempo')
axes[0, 0].set_ylabel('Residuo')

axes[0, 1].hist(resid, bins=20, color='steelblue', edgecolor='white', density=True)
x = np.linspace(resid.min(), resid.max(), 100)
axes[0, 1].plot(x, norm.pdf(x, resid.mean(), resid.std()), 'r-', linewidth=2)
axes[0, 1].set_title('Histograma dos Residuos')

max_lag = 24
n_r = len(resid)
acf = np.array([np.corrcoef(resid[:n_r-k], resid[k:])[0, 1] for k in range(max_lag + 1)])
acf[0] = 1.0
ci = 1.96 / np.sqrt(n_r)
axes[1, 0].bar(range(max_lag + 1), acf, width=0.3, color='steelblue')
axes[1, 0].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[1, 0].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[1, 0].set_title('ACF dos Residuos')

probplot(resid, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot')

plt.suptitle('Diagnostico de Residuos - ETS(A,A,A)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Previsao

In [ ]:
# Previsao 24 meses a frente com ETS(A,A,A)
fc = res_aaa.forecast(steps=24)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue')
ax.plot(range(len(y_airline)), res_aaa.fittedvalues, label='Ajustado',
        color='darkorange', linestyle='--', alpha=0.7)
ax.plot(range(len(y_airline), len(y_airline) + 24), fc,
        label='Previsao ETS(A,A,A)', color='crimson', linewidth=2)
ax.axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('ETS(A,A,A) - Previsao 24 Meses')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('Previsoes (cada 6 meses):')
for i in range(0, 24, 6):
    print(f'  Passo {i+1:2d}: {fc[i]:.1f}')

---

## Exercicio 1 - SOLUCAO: Identificar o modelo ETS adequado

Usando o dataset `airline.csv`:

1. Ajuste os modelos ETS(A,N,N), ETS(A,A,N), ETS(A,A,A), ETS(M,A,M) e ETS(M,Ad,M)
2. Compare AIC e BIC de cada modelo
3. Qual modelo melhor captura a sazonalidade multiplicativa dos dados?
4. Plote os ajustes sobrepostos e justifique sua escolha

**Dica:** A amplitude da sazonalidade do airline cresce com o nivel - isso sugere sazonalidade multiplicativa!

In [ ]:
# Exercicio 1 - SOLUCAO
# Passo 1: Ajustar todos os 5 modelos candidatos

modelos_ex1 = {}

# ETS(A,N,N) - SES
model = ETS(error='A', trend='N', seasonal='N')
modelos_ex1['ETS(A,N,N)'] = model.fit(y_airline)

# ETS(A,A,N) - Holt
model = ETS(error='A', trend='A', seasonal='N')
modelos_ex1['ETS(A,A,N)'] = model.fit(y_airline)

# ETS(A,A,A) - Holt-Winters aditivo
model = ETS(error='A', trend='A', seasonal='A', seasonal_period=12)
modelos_ex1['ETS(A,A,A)'] = model.fit(y_airline)

# ETS(M,A,M) - Holt-Winters multiplicativo
model = ETS(error='M', trend='A', seasonal='M', seasonal_period=12)
modelos_ex1['ETS(M,A,M)'] = model.fit(y_airline)

# ETS(M,Ad,M) - Multiplicativo com tendencia amortecida
model = ETS(error='M', trend='A', seasonal='M', seasonal_period=12, damped=True)
modelos_ex1['ETS(M,Ad,M)'] = model.fit(y_airline)

print(f'{"Modelo":<15} {"alpha":>8} {"beta":>8} {"gamma":>8} {"phi":>8} {"AIC":>10} {"BIC":>10} {"AICc":>10}')
print('-' * 80)

for nome, res in modelos_ex1.items():
    alpha = f'{res.alpha:.4f}'
    beta = f'{res.beta:.4f}' if res.beta is not None else '   -   '
    gamma = f'{res.gamma:.4f}' if res.gamma is not None else '   -   '
    phi = f'{res.phi:.4f}' if res.phi is not None else '   -   '
    print(f'{nome:<15} {alpha:>8} {beta:>8} {gamma:>8} {phi:>8} {res.aic:>10.2f} {res.bic:>10.2f} {res.aicc:>10.2f}')

In [ ]:
# Passo 2: Identificar o melhor modelo
melhor_nome = min(modelos_ex1, key=lambda k: modelos_ex1[k].aic)
melhor_res = modelos_ex1[melhor_nome]

print(f'Melhor modelo por AIC: {melhor_nome} (AIC = {melhor_res.aic:.2f})')
print(f'Melhor modelo por BIC: {min(modelos_ex1, key=lambda k: modelos_ex1[k].bic)}')
print(f'Melhor modelo por AICc: {min(modelos_ex1, key=lambda k: modelos_ex1[k].aicc)}')

# Metricas de erro in-sample
print(f'\n{"Modelo":<15} {"RMSE":>10} {"MAE":>10} {"MAPE(%)":>10}')
print('-' * 50)
for nome, res in modelos_ex1.items():
    residuos = res.resid
    rmse = np.sqrt(np.mean(residuos**2))
    mae = np.mean(np.abs(residuos))
    mape = np.mean(np.abs(residuos / y_airline)) * 100
    print(f'{nome:<15} {rmse:>10.2f} {mae:>10.2f} {mape:>10.2f}')

In [ ]:
# Passo 3: Plotar ajustes sobrepostos
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

cores = ['darkorange', 'green', 'purple', 'crimson', 'brown']

# Painel superior: todos os ajustes
axes[0].plot(y_airline, label='Observado', color='steelblue', linewidth=2)
for (nome, res), cor in zip(modelos_ex1.items(), cores):
    axes[0].plot(res.fittedvalues, label=f'{nome} (AIC={res.aic:.0f})',
                 color=cor, linewidth=1, linestyle='--', alpha=0.8)
axes[0].set_title('Comparacao de Modelos ETS - Airline Passengers')
axes[0].set_ylabel('Passageiros')
axes[0].legend(fontsize=9)

# Painel inferior: residuos dos dois melhores
axes[1].plot(modelos_ex1['ETS(A,A,A)'].resid, label='Residuos ETS(A,A,A)',
             color='purple', alpha=0.7)
axes[1].plot(modelos_ex1['ETS(M,A,M)'].resid, label='Residuos ETS(M,A,M)',
             color='crimson', alpha=0.7)
axes[1].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[1].set_title('Comparacao de Residuos: ETS(A,A,A) vs ETS(M,A,M)')
axes[1].set_ylabel('Residuo')
axes[1].set_xlabel('Observacao')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nAnalise:')
print('- Os modelos sem sazonalidade (A,N,N e A,A,N) tem AIC muito superior (pior).')
print('- ETS(M,A,M) captura melhor a sazonalidade multiplicativa do airline.')
print('- A amplitude da sazonalidade cresce com o nivel, confirmando o padrao multiplicativo.')
print('- Os residuos do ETS(M,A,M) sao mais homoscedasticos que os do ETS(A,A,A).')

In [ ]:
# Passo 4: Previsao com o melhor modelo e diagnostico
best = modelos_ex1[melhor_nome]
fc_best = best.forecast(steps=24)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Previsao
axes[0].plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue')
axes[0].plot(range(len(y_airline)), best.fittedvalues, label='Ajustado',
             color='darkorange', linestyle='--', alpha=0.7)
axes[0].plot(range(len(y_airline), len(y_airline) + 24), fc_best,
             label=f'Previsao {melhor_nome}', color='crimson', linewidth=2)
axes[0].axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)
axes[0].set_title(f'{melhor_nome} - Previsao 24 Meses')
axes[0].legend(fontsize=9)

# ACF residuos
resid_best = best.resid
n_r = len(resid_best)
acf_vals = np.array([np.corrcoef(resid_best[:n_r-k], resid_best[k:])[0, 1] for k in range(25)])
acf_vals[0] = 1.0
ci = 1.96 / np.sqrt(n_r)
axes[1].bar(range(25), acf_vals, width=0.3, color='steelblue')
axes[1].axhline(ci, color='red', linestyle='--', alpha=0.7)
axes[1].axhline(-ci, color='red', linestyle='--', alpha=0.7)
axes[1].set_title(f'ACF dos Residuos - {melhor_nome}')
axes[1].set_xlabel('Lag')

plt.tight_layout()
plt.show()

### Conclusao do Exercicio 1

O modelo **ETS(M,A,M)** e o mais adequado para o dataset airline, pois:

1. **Menor AIC/BIC/AICc** entre os candidatos
2. **Sazonalidade multiplicativa** captura o padrao de amplitude crescente
3. **Residuos homoscedasticos**: variancia mais constante ao longo do tempo
4. O ETS(M,Ad,M) com damping tem AIC similar, indicando que a tendencia pode ser ligeiramente amortecida

A sazonalidade multiplicativa e adequada porque a amplitude dos picos sazonais cresce proporcionalmente ao nivel da serie (mais passageiros no verao, e esse excesso cresce com o tempo).

---

## Exercicio 2 - SOLUCAO: ETS no dataset sintetico

Usando o dataset `ets_synthetic.csv`:

1. Plote a serie e identifique visualmente tendencia e tipo de sazonalidade
2. Ajuste ETS(A,A,A) e ETS(M,A,M) e compare os resultados
3. Analise os residuos do melhor modelo
4. O dataset foi gerado como ETS(M,A,M) - seu modelo conseguiu identificar isso?

In [ ]:
# Exercicio 2 - SOLUCAO
# Passo 1: Visualizacao exploratoria
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(y_synth, color='darkorange', linewidth=1)
axes[0].set_title('Serie Sintetica ETS - Identificacao Visual')
axes[0].set_ylabel('Valor')
axes[0].set_xlabel('Observacao')

# Amplitude sazonal: diferenca entre max e min por ano
n_anos = len(y_synth) // 12
amplitudes = []
niveis = []
for a in range(n_anos):
    bloco = y_synth[a*12:(a+1)*12]
    amplitudes.append(bloco.max() - bloco.min())
    niveis.append(bloco.mean())

axes[1].scatter(niveis, amplitudes, color='darkorange', s=60, edgecolors='black')
# Regressao linear para mostrar relacao
z = np.polyfit(niveis, amplitudes, 1)
p = np.poly1d(z)
x_reg = np.linspace(min(niveis), max(niveis), 50)
axes[1].plot(x_reg, p(x_reg), 'r--', linewidth=2)
axes[1].set_title('Amplitude Sazonal vs Nivel Medio Anual')
axes[1].set_xlabel('Nivel medio anual')
axes[1].set_ylabel('Amplitude sazonal')

plt.tight_layout()
plt.show()

print('Observacoes:')
print('- A serie apresenta tendencia crescente clara.')
print('- A amplitude sazonal cresce com o nivel -> sazonalidade MULTIPLICATIVA.')
print(f'- Correlacao amplitude vs nivel: {np.corrcoef(niveis, amplitudes)[0,1]:.4f}')

In [ ]:
# Passo 2: Ajustar ETS(A,A,A) vs ETS(M,A,M)
model_aaa_s = ETS(error='A', trend='A', seasonal='A', seasonal_period=12)
res_aaa_s = model_aaa_s.fit(y_synth)

model_mam_s = ETS(error='M', trend='A', seasonal='M', seasonal_period=12)
res_mam_s = model_mam_s.fit(y_synth)

print(f'{"Modelo":<15} {"alpha":>8} {"beta":>8} {"gamma":>8} {"AIC":>10} {"BIC":>10} {"AICc":>10}')
print('-' * 70)
for nome, res in [('ETS(A,A,A)', res_aaa_s), ('ETS(M,A,M)', res_mam_s)]:
    print(f'{nome:<15} {res.alpha:>8.4f} {res.beta:>8.4f} {res.gamma:>8.4f} {res.aic:>10.2f} {res.bic:>10.2f} {res.aicc:>10.2f}')

# Metricas de erro
print(f'\n{"Modelo":<15} {"RMSE":>10} {"MAE":>10} {"MAPE(%)":>10}')
print('-' * 50)
for nome, res in [('ETS(A,A,A)', res_aaa_s), ('ETS(M,A,M)', res_mam_s)]:
    r = res.resid
    rmse_val = np.sqrt(np.mean(r**2))
    mae_val = np.mean(np.abs(r))
    mape_val = np.mean(np.abs(r / y_synth)) * 100
    print(f'{nome:<15} {rmse_val:>10.4f} {mae_val:>10.4f} {mape_val:>10.4f}')

melhor_synth = 'ETS(M,A,M)' if res_mam_s.aic < res_aaa_s.aic else 'ETS(A,A,A)'
print(f'\nMelhor modelo: {melhor_synth}')

In [ ]:
# Passo 3: Diagnostico de residuos do melhor modelo
res_melhor_s = res_mam_s if melhor_synth == 'ETS(M,A,M)' else res_aaa_s
resid_s = res_melhor_s.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(resid_s, color='darkorange', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Residuos ao Longo do Tempo')
axes[0, 0].set_ylabel('Residuo')

axes[0, 1].hist(resid_s, bins=20, color='darkorange', edgecolor='white', density=True)
x_hist = np.linspace(resid_s.min(), resid_s.max(), 100)
axes[0, 1].plot(x_hist, norm.pdf(x_hist, resid_s.mean(), resid_s.std()), 'r-', linewidth=2)
axes[0, 1].set_title('Histograma dos Residuos')

n_rs = len(resid_s)
acf_s = np.array([np.corrcoef(resid_s[:n_rs-k], resid_s[k:])[0, 1] for k in range(25)])
acf_s[0] = 1.0
ci_s = 1.96 / np.sqrt(n_rs)
axes[1, 0].bar(range(25), acf_s, width=0.3, color='darkorange')
axes[1, 0].axhline(ci_s, color='red', linestyle='--', alpha=0.7)
axes[1, 0].axhline(-ci_s, color='red', linestyle='--', alpha=0.7)
axes[1, 0].set_title('ACF dos Residuos')

probplot(resid_s, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot')

plt.suptitle(f'Diagnostico de Residuos - {melhor_synth} (Serie Sintetica)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Passo 4: Conclusao - O modelo recuperou o processo gerador?
print('=' * 60)
print('RESULTADO: Identificacao do Modelo Gerador')
print('=' * 60)
print(f'\nProcesso gerador real:  ETS(M,A,M)')
print(f'Modelo selecionado:     {melhor_synth}')
print(f'\nO modelo {"CONSEGUIU" if melhor_synth == "ETS(M,A,M)" else "NAO conseguiu"} recuperar o processo gerador!')
print(f'\nParametros estimados do {melhor_synth}:')
print(f'  alpha = {res_melhor_s.alpha:.4f}')
print(f'  beta  = {res_melhor_s.beta:.4f}')
print(f'  gamma = {res_melhor_s.gamma:.4f}')
print(f'  AIC   = {res_melhor_s.aic:.2f}')
print(f'  BIC   = {res_melhor_s.bic:.2f}')

### Conclusao do Exercicio 2

O modelo **ETS(M,A,M)** e corretamente identificado como o melhor modelo para a serie sintetica:

1. **Menor AIC/BIC** comparado ao ETS(A,A,A)
2. **Residuos mais bem comportados**: distribuicao mais proxima da normal
3. **Recupera o processo gerador**: a serie foi gerada como ETS(M,A,M) e o modelo identificou corretamente esta especificacao
4. A analise visual ja indicava sazonalidade multiplicativa (amplitude crescente com o nivel)

---

## 9. Salvando Resultados para Comparacao

In [ ]:
# Salvar coeficientes em outputs/ets_coefficients.json
coeficientes = {}

# Coeficientes do Exercicio 1 (airline)
for nome, res in modelos_ex1.items():
    key = f'airline_{nome}'
    entry = {
        'alpha': round(res.alpha, 6),
        'sigma2': round(res.sigma2, 6),
        'aic': round(res.aic, 4),
        'bic': round(res.bic, 4),
        'aicc': round(res.aicc, 4),
        'loglike': round(res.loglik, 4),
    }
    if res.beta is not None:
        entry['beta'] = round(res.beta, 6)
    if res.gamma is not None:
        entry['gamma'] = round(res.gamma, 6)
    if res.phi is not None:
        entry['phi'] = round(res.phi, 6)
    entry['l0'] = round(res.l0, 4)
    if res.b0 is not None:
        entry['b0'] = round(res.b0, 4)
    # Metricas de erro
    r = res.resid
    entry['rmse'] = round(float(np.sqrt(np.mean(r**2))), 4)
    entry['mae'] = round(float(np.mean(np.abs(r))), 4)
    entry['mape'] = round(float(np.mean(np.abs(r / y_airline)) * 100), 4)
    coeficientes[key] = entry

# Coeficientes do Exercicio 2 (sintetico)
for nome, res in [('ETS(A,A,A)', res_aaa_s), ('ETS(M,A,M)', res_mam_s)]:
    key = f'synthetic_{nome}'
    entry = {
        'alpha': round(res.alpha, 6),
        'sigma2': round(res.sigma2, 6),
        'aic': round(res.aic, 4),
        'bic': round(res.bic, 4),
        'aicc': round(res.aicc, 4),
        'loglike': round(res.loglik, 4),
    }
    if res.beta is not None:
        entry['beta'] = round(res.beta, 6)
    if res.gamma is not None:
        entry['gamma'] = round(res.gamma, 6)
    entry['l0'] = round(res.l0, 4)
    if res.b0 is not None:
        entry['b0'] = round(res.b0, 4)
    r = res.resid
    entry['rmse'] = round(float(np.sqrt(np.mean(r**2))), 4)
    entry['mae'] = round(float(np.mean(np.abs(r))), 4)
    entry['mape'] = round(float(np.mean(np.abs(r / y_synth)) * 100), 4)
    coeficientes[key] = entry

# Salvar
output_path = os.path.join(OUTPUT_DIR, 'ets_coefficients.json')
with open(output_path, 'w') as f:
    json.dump(coeficientes, f, indent=2)

print(f'Coeficientes salvos em: {output_path}')
print(f'Total de modelos salvos: {len(coeficientes)}')
print('\nResumo dos modelos salvos:')
for key in coeficientes:
    print(f'  {key}: AIC={coeficientes[key]["aic"]:.2f}')

---

## Conclusao

Neste notebook aprendemos:

1. **Taxonomia ETS**: Os 30 modelos possiveis e quando usar cada um
2. **Progressao de complexidade**: SES -> Holt -> Holt-Winters
3. **Identificacao de modelo**: AIC/BIC como criterios de selecao
4. **Sazonalidade multiplicativa**: quando a amplitude sazonal cresce com o nivel, modelos com componente multiplicativo (M) sao preferidos
5. **Diagnostico**: residuos devem ser ruido branco (sem autocorrelacao, distribuicao normal)

**Proximo notebook:** Holt-Winters em detalhe (aditivo, multiplicativo, damped trend)